<div style="display: flex; align-items: center;">

  <!-- Logos -->
  <div style="white-space: nowrap;">
    <img 
      src="https://www.upc.edu/comunicacio/ca/identitat/descarrega-arxius-grafics/fitxers-marca-principal/upc-positiu-p3005.png" 
      width="300"
      style="vertical-align: middle;"
    >
    <img 
      src="https://www.hipotecalowcost.com/wp-content/uploads/2019/08/Logo-CaixaBank.png" 
      width="200"
      style="vertical-align: middle;"
    >
  </div>

  <!-- Texto -->
  <div style="margin-left: auto; margin-right: 10px; text-align: right;">
      <p style="margin: 0;"><b>CaixaBank · Advanced Analytics Program</b></p>
      <p style="margin: 0;"><b>Model Risk & Data Science Training</b></p>
      <p style="margin: 0;">Automatización de procesos</p>
  </div>

</div>

# **Generación masiva de informes Excel parametrizados**
## *Un dataset maestro → múltiples workbooks*

Construiremos un sistema que genere automáticamente un Excel por cliente, región o periodo.

```text
dataset maestro
      ↓
validación
      ↓
segmentación
      ↓
generador reusable
      ↓
Datos + Resumen + Dashboard
      ↓
guardado
      ↓
reapertura
      ↓
auditoría
```

## Objetivos

- Validar una fuente maestra.
- Segmentar datos.
- Parametrizar informes.
- Crear un workbook por grupo.
- Añadir KPIs y gráficos.
- Sanitizar nombres de fichero.
- Capturar errores por informe.
- Continuar un lote aunque falle una salida.
- Crear auditoría y KPIs del lote.
- Reutilizar el generador con distintos criterios.

## 1. Preparación

In [ ]:
from pathlib import Path
import shutil
import random
import time
import re

import pandas as pd
import matplotlib.pyplot as plt

try:
    from openpyxl import Workbook, load_workbook
    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.chart import BarChart, Reference
    from openpyxl.utils import get_column_letter
    OPENPYXL_DISPONIBLE = True
except ImportError:
    OPENPYXL_DISPONIBLE = False

print("openpyxl disponible:", OPENPYXL_DISPONIBLE)

## 2. Crear estructura

In [ ]:
raiz = Path("demo_informes_excel_masivos")

if raiz.exists():
    shutil.rmtree(raiz)

for carpeta in [
    raiz / "data",
    raiz / "outputs",
    raiz / "logs"
]:
    carpeta.mkdir(parents=True, exist_ok=True)

print(raiz.resolve())

## 3. Crear dataset maestro

In [ ]:
random.seed(42)

clientes = [
    "Cliente Norte",
    "Cliente Sur",
    "Cliente Este",
    "Cliente Oeste",
    "Cliente Centro"
]

regiones = [
    "Norte",
    "Sur",
    "Este",
    "Oeste",
    "Centro"
]

productos = [
    "Producto A",
    "Producto B",
    "Producto C",
    "Producto D"
]

periodos = [
    "2026-Q1",
    "2026-Q2"
]

filas = []
id_venta = 1

for cliente, region in zip(clientes, regiones):
    for periodo in periodos:
        for _ in range(40):
            producto = random.choice(productos)
            unidades = random.randint(1, 20)
            precio = random.choice([25, 40, 60, 100])
            descuento = random.choice([0, 0.05, 0.10, 0.15])

            filas.append({
                "ID": id_venta,
                "Cliente": cliente,
                "Region": region,
                "Periodo": periodo,
                "Producto": producto,
                "Unidades": unidades,
                "Precio": precio,
                "Descuento": descuento,
                "Importe": unidades * precio * (1 - descuento)
            })

            id_venta += 1

df = pd.DataFrame(filas)

ruta_maestra = raiz / "data" / "ventas_maestro.csv"
df.to_csv(ruta_maestra, index=False)

df.head()

## 3.1 Ver dimensiones

In [ ]:
print("Filas:", len(df))
print("Clientes:", df["Cliente"].nunique())
print("Periodos:", df["Periodo"].nunique())
print(
    "Informes cliente-periodo esperados:",
    len(
        df[
            ["Cliente", "Periodo"]
        ].drop_duplicates()
    )
)

## 3.2 Visualizar distribución

In [ ]:
distribucion = (
    df.groupby(["Cliente", "Periodo"])
    .size()
    .unstack(fill_value=0)
)

distribucion.plot(
    kind="bar",
    figsize=(10, 5)
)

plt.ylabel("Operaciones")
plt.title("Operaciones por cliente y periodo")
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

## 4. Introducir inconsistencias antes de generar

In [ ]:
df_problematico = df.copy()

df_problematico.loc[5, "Cliente"] = None
df_problematico.loc[20, "Unidades"] = -3
df_problematico.loc[80, "Descuento"] = 1.5

df_problematico.loc[[5, 20, 80]]

## 4.1 Validar fuente maestra

In [ ]:
COLUMNAS_OBLIGATORIAS = [
    "ID", "Cliente", "Region", "Periodo",
    "Producto", "Unidades", "Precio",
    "Descuento", "Importe"
]

def validar_dataset(datos):
    errores = []

    faltantes = [
        c for c in COLUMNAS_OBLIGATORIAS
        if c not in datos.columns
    ]

    if faltantes:
        return [f"Faltan columnas: {faltantes}"]

    if datos["Cliente"].isna().any():
        errores.append("Hay clientes ausentes.")

    if (datos["Unidades"] <= 0).any():
        errores.append("Hay unidades no positivas.")

    if (
        (datos["Descuento"] < 0)
        | (datos["Descuento"] > 1)
    ).any():
        errores.append("Hay descuentos fuera de rango.")

    return errores

validar_dataset(df_problematico)

## 4.2 Corregir explícitamente

In [ ]:
df_limpio = df_problematico.copy()

df_limpio["Cliente"] = (
    df_limpio["Cliente"]
    .fillna("Cliente desconocido")
)

df_limpio.loc[
    df_limpio["Unidades"] <= 0,
    "Unidades"
] = 1

df_limpio["Descuento"] = (
    df_limpio["Descuento"]
    .clip(0, 1)
)

df_limpio["Importe"] = (
    df_limpio["Unidades"]
    * df_limpio["Precio"]
    * (1 - df_limpio["Descuento"])
)

print(validar_dataset(df_limpio))

## 5. Sanitizar nombres de fichero

In [ ]:
def nombre_archivo_seguro(texto):
    texto = str(texto).strip()
    texto = re.sub(r"\s+", "_", texto)
    texto = re.sub(r"[^A-Za-z0-9_-]+", "", texto)
    return texto

print(
    nombre_archivo_seguro(
        "Cliente Norte / 2026-Q1"
    )
)

### Interpretación

Los valores de negocio no deberían convertirse directamente en nombres físicos sin normalización.

## 6. Funciones de estilo

In [ ]:
if OPENPYXL_DISPONIBLE:

    COLOR_CABECERA = "1F4E78"
    COLOR_KPI = "D9EAF7"

    lado = Side(
        style="thin",
        color="BFBFBF"
    )

    borde = Border(
        left=lado,
        right=lado,
        top=lado,
        bottom=lado
    )

    def aplicar_cabecera(
        worksheet,
        fila=1
    ):
        for celda in worksheet[fila]:
            celda.font = Font(
                bold=True,
                color="FFFFFF"
            )
            celda.fill = PatternFill(
                fill_type="solid",
                fgColor=COLOR_CABECERA
            )
            celda.alignment = Alignment(
                horizontal="center"
            )
            celda.border = borde

    def ajustar_anchos(
        worksheet,
        limite=30
    ):
        for columna in worksheet.columns:
            letra = get_column_letter(
                columna[0].column
            )
            maximo = max(
                len(str(celda.value))
                if celda.value is not None
                else 0
                for celda in columna
            )
            worksheet.column_dimensions[
                letra
            ].width = min(
                maximo + 2,
                limite
            )

## 7. KPIs y resumen por producto

In [ ]:
def calcular_kpis_segmento(datos):
    return {
        "Facturacion":
            float(datos["Importe"].sum()),

        "Operaciones":
            int(len(datos)),

        "Unidades":
            int(datos["Unidades"].sum()),

        "Ticket_medio":
            float(datos["Importe"].mean()),

        "Productos":
            int(datos["Producto"].nunique())
    }

def resumen_producto(datos):
    return (
        datos.groupby(
            "Producto",
            as_index=False
        )
        .agg(
            Facturacion=("Importe", "sum"),
            Operaciones=("ID", "count"),
            Unidades=("Unidades", "sum")
        )
        .sort_values(
            "Facturacion",
            ascending=False
        )
    )

## 7.1 Probar con un segmento

In [ ]:
ejemplo_segmento = df_limpio[
    (df_limpio["Cliente"] == "Cliente Norte")
    & (df_limpio["Periodo"] == "2026-Q1")
]

print(
    calcular_kpis_segmento(
        ejemplo_segmento
    )
)

display(
    resumen_producto(
        ejemplo_segmento
    )
)

## 8. Función para escribir un DataFrame

In [ ]:
if OPENPYXL_DISPONIBLE:

    def escribir_dataframe(
        dataframe,
        worksheet
    ):
        worksheet.append(
            list(dataframe.columns)
        )

        for fila in dataframe.itertuples(
            index=False,
            name=None
        ):
            worksheet.append(
                list(fila)
            )

        return worksheet

## 9. Generador reusable de workbook

In [ ]:
if OPENPYXL_DISPONIBLE:

    def crear_informe_excel(
        datos_segmento,
        cliente,
        periodo,
        ruta_salida
    ):
        columnas_necesarias = {
            "ID", "Cliente", "Region", "Periodo",
            "Producto", "Unidades", "Precio",
            "Descuento", "Importe"
        }

        faltantes = (
            columnas_necesarias
            - set(datos_segmento.columns)
        )

        if faltantes:
            raise ValueError(
                f"Faltan columnas: {sorted(faltantes)}"
            )

        wb = Workbook()

        ws_datos = wb.active
        ws_datos.title = "Datos"

        ws_resumen = wb.create_sheet(
            "Resumen"
        )

        ws_dashboard = wb.create_sheet(
            "Dashboard"
        )

        # Datos
        escribir_dataframe(
            datos_segmento,
            ws_datos
        )

        aplicar_cabecera(
            ws_datos
        )

        ajustar_anchos(
            ws_datos
        )

        ws_datos.freeze_panes = "A2"

        for fila in range(
            2,
            ws_datos.max_row + 1
        ):
            ws_datos.cell(
                fila, 7
            ).number_format = '#,##0.00 €'

            ws_datos.cell(
                fila, 8
            ).number_format = '0%'

            ws_datos.cell(
                fila, 9
            ).number_format = '#,##0.00 €'

        # Resumen
        resumen = resumen_producto(
            datos_segmento
        )

        escribir_dataframe(
            resumen,
            ws_resumen
        )

        aplicar_cabecera(
            ws_resumen
        )

        ajustar_anchos(
            ws_resumen
        )

        # Dashboard
        ws_dashboard.merge_cells(
            "A1:F2"
        )

        ws_dashboard["A1"] = (
            f"Informe - {cliente} - {periodo}"
        )

        ws_dashboard["A1"].font = Font(
            bold=True,
            size=18,
            color="FFFFFF"
        )

        ws_dashboard["A1"].fill = PatternFill(
            fill_type="solid",
            fgColor=COLOR_CABECERA
        )

        ws_dashboard["A1"].alignment = Alignment(
            horizontal="center"
        )

        kpis = calcular_kpis_segmento(
            datos_segmento
        )

        posiciones = [
            ("A4", "B4"),
            ("D4", "E4"),
            ("A7", "B7"),
            ("D7", "E7"),
            ("A10", "B10")
        ]

        for (
            nombre,
            valor
        ), (
            c_nombre,
            c_valor
        ) in zip(
            kpis.items(),
            posiciones
        ):
            ws_dashboard[c_nombre] = nombre
            ws_dashboard[c_valor] = valor

            ws_dashboard[c_nombre].font = Font(
                bold=True
            )

            ws_dashboard[c_nombre].fill = PatternFill(
                fill_type="solid",
                fgColor=COLOR_KPI
            )

            ws_dashboard[c_valor].font = Font(
                bold=True,
                size=14
            )

        ws_dashboard["B4"].number_format = '#,##0.00 €'
        ws_dashboard["E7"].number_format = '#,##0.00 €'

        # Tabla resumen para gráfico
        inicio = 14

        ws_dashboard.cell(
            inicio,
            1,
            "Producto"
        )

        ws_dashboard.cell(
            inicio,
            2,
            "Facturacion"
        )

        for offset, fila in enumerate(
            resumen.itertuples(
                index=False
            ),
            start=1
        ):
            ws_dashboard.cell(
                inicio + offset,
                1,
                fila.Producto
            )

            ws_dashboard.cell(
                inicio + offset,
                2,
                fila.Facturacion
            )

        aplicar_cabecera(
            ws_dashboard,
            fila=inicio
        )

        grafico = BarChart()
        grafico.title = "Facturación por producto"

        datos_chart = Reference(
            ws_dashboard,
            min_col=2,
            min_row=inicio,
            max_row=inicio + len(resumen)
        )

        categorias = Reference(
            ws_dashboard,
            min_col=1,
            min_row=inicio + 1,
            max_row=inicio + len(resumen)
        )

        grafico.add_data(
            datos_chart,
            titles_from_data=True
        )

        grafico.set_categories(
            categorias
        )

        grafico.height = 8
        grafico.width = 13

        ws_dashboard.add_chart(
            grafico,
            "D14"
        )

        ws_dashboard.sheet_view.showGridLines = False

        for columna in range(1, 7):
            ws_dashboard.column_dimensions[
                get_column_letter(columna)
            ].width = 18

        ruta_salida = Path(
            ruta_salida
        )

        ruta_salida.parent.mkdir(
            parents=True,
            exist_ok=True
        )

        wb.save(
            ruta_salida
        )

        return {
            "ruta":
                ruta_salida,
            "filas":
                len(datos_segmento),
            "kpis":
                kpis
        }

## 9.1 Generar un primer informe

In [ ]:
if OPENPYXL_DISPONIBLE:

    ruta_demo = (
        raiz
        / "outputs"
        / "Cliente_Norte_2026-Q1.xlsx"
    )

    resultado_demo = crear_informe_excel(
        ejemplo_segmento,
        "Cliente Norte",
        "2026-Q1",
        ruta_demo
    )

    resultado_demo

## 9.2 Reabrir y validar

In [ ]:
if OPENPYXL_DISPONIBLE:

    wb_demo = load_workbook(
        ruta_demo
    )

    assert {
        "Datos",
        "Resumen",
        "Dashboard"
    }.issubset(
        set(wb_demo.sheetnames)
    )

    assert len(
        wb_demo["Dashboard"]._charts
    ) >= 1

    print(
        "Informe individual validado."
    )

## 10. Crear segmentos

In [ ]:
segmentos = (
    df_limpio[
        ["Cliente", "Periodo"]
    ]
    .drop_duplicates()
    .sort_values(
        ["Cliente", "Periodo"]
    )
    .reset_index(drop=True)
)

segmentos

## 11. Generación masiva

In [ ]:
if OPENPYXL_DISPONIBLE:

    auditoria = []

    for _, segmento in segmentos.iterrows():

        cliente = segmento["Cliente"]
        periodo = segmento["Periodo"]

        datos_segmento = df_limpio[
            (df_limpio["Cliente"] == cliente)
            & (df_limpio["Periodo"] == periodo)
        ].copy()

        nombre = (
            f"{nombre_archivo_seguro(cliente)}_"
            f"{nombre_archivo_seguro(periodo)}.xlsx"
        )

        ruta = (
            raiz
            / "outputs"
            / nombre
        )

        inicio = time.perf_counter()

        try:
            crear_informe_excel(
                datos_segmento,
                cliente,
                periodo,
                ruta
            )

            estado = "ok"
            mensaje = ""

        except Exception as error:
            estado = "error"
            mensaje = str(error)

        auditoria.append({
            "Cliente": cliente,
            "Periodo": periodo,
            "Filas": len(datos_segmento),
            "Ruta": str(ruta),
            "Estado": estado,
            "Mensaje": mensaje,
            "Segundos":
                time.perf_counter()
                - inicio
        })

    auditoria = pd.DataFrame(
        auditoria
    )

    auditoria

## 11.1 Visualizar estado del lote

In [ ]:
if OPENPYXL_DISPONIBLE:

    estado = auditoria[
        "Estado"
    ].value_counts()

    plt.figure(
        figsize=(7, 5)
    )

    plt.bar(
        estado.index,
        estado.values
    )

    plt.ylabel("Informes")
    plt.title("Estado de generación masiva")
    plt.tight_layout()
    plt.show()

## 12. Validar outputs físicos

In [ ]:
if OPENPYXL_DISPONIBLE:

    validacion = pd.DataFrame({
        "Ruta":
            auditoria["Ruta"],

        "Existe":
            auditoria["Ruta"].apply(
                lambda x:
                    Path(x).exists()
            ),

        "Bytes":
            auditoria["Ruta"].apply(
                lambda x:
                    Path(x).stat().st_size
                    if Path(x).exists()
                    else 0
            )
    })

    validacion

## 12.1 Validaciones finales

In [ ]:
if OPENPYXL_DISPONIBLE:

    assert validacion[
        "Existe"
    ].all()

    assert (
        validacion[
            "Bytes"
        ] > 0
    ).all()

    print(
        "Todos los workbooks existen y tienen contenido."
    )

## 13. Provocar un error individual

In [ ]:
if OPENPYXL_DISPONIBLE:

    segmento_error = (
        ejemplo_segmento
        .drop(
            columns=["Producto"]
        )
    )

    try:
        crear_informe_excel(
            segmento_error,
            "Cliente Error",
            "2026-Q1",
            raiz
            / "outputs"
            / "cliente_error.xlsx"
        )

    except Exception as error:
        print(
            type(error).__name__,
            "->",
            error
        )

### Interpretación

El error es local a un informe.

En producción debemos registrarlo y continuar con el resto del lote.

## 14. Guardar auditoría

In [ ]:
if OPENPYXL_DISPONIBLE:

    ruta_auditoria = (
        raiz
        / "logs"
        / "auditoria_generacion.csv"
    )

    auditoria.to_csv(
        ruta_auditoria,
        index=False
    )

    print(
        ruta_auditoria
    )

## 15. KPIs del lote

In [ ]:
if OPENPYXL_DISPONIBLE:

    kpis_lote = {
        "Informes":
            len(auditoria),

        "Correctos":
            int(
                (
                    auditoria["Estado"]
                    == "ok"
                ).sum()
            ),

        "Errores":
            int(
                (
                    auditoria["Estado"]
                    == "error"
                ).sum()
            ),

        "Tiempo_total":
            float(
                auditoria["Segundos"].sum()
            ),

        "Tiempo_medio":
            float(
                auditoria["Segundos"].mean()
            )
    }

    kpis_lote[
        "Tasa_exito"
    ] = (
        kpis_lote[
            "Correctos"
        ]
        / kpis_lote[
            "Informes"
        ]
    )

    kpis_lote

## 16. Segmentación genérica

In [ ]:
def obtener_segmentos(
    dataframe,
    columnas
):
    return (
        dataframe[
            columnas
        ]
        .drop_duplicates()
        .sort_values(columnas)
        .reset_index(drop=True)
    )

obtener_segmentos(
    df_limpio,
    ["Region", "Periodo"]
)

### Interpretación

Podemos reutilizar el mismo generador para:

- cliente;
- región;
- departamento;
- periodo;
- cliente-periodo;
- cualquier combinación de dimensiones.

## 17. Arquitectura reusable

In [ ]:
pd.DataFrame({
    "componente": [
        "ventas_maestro.csv",
        "validar_dataset()",
        "obtener_segmentos()",
        "crear_informe_excel()",
        "outputs/",
        "auditoria_generacion.csv"
    ],
    "responsabilidad": [
        "Fuente",
        "Calidad",
        "Segmentación",
        "Generación",
        "Salidas",
        "Trazabilidad"
    ]
})

## 18. Mini-proyecto integrado

Construye un dataset con:

- 10 clientes;
- 4 regiones;
- 4 periodos;
- 1.000 operaciones.

Genera un informe por cliente-periodo con:

1. `Datos`;
2. `Resumen`;
3. `Dashboard`;
4. 5 KPIs;
5. tabla resumen;
6. gráfico;
7. estilos;
8. formatos numéricos;
9. nombre seguro;
10. guardado;
11. reapertura;
12. validación.

El lote debe:

13. capturar errores;
14. continuar;
15. crear auditoría;
16. medir tiempos;
17. calcular tasa de éxito.

## 19. Ejercicios finales

1. Genera un dataset maestro.
2. Valida columnas.
3. Introduce tres inconsistencias.
4. Corrige antes de generar.
5. Sanitiza nombres.
6. Segmenta por cliente.
7. Segmenta por cliente-periodo.
8. Calcula KPIs.
9. Resume por producto.
10. Genera un workbook individual.
11. Añade dashboard.
12. Añade gráfico.
13. Genera 20 workbooks.
14. Provoca un error individual.
15. Continúa el lote.
16. Genera auditoría.
17. Valida archivos.
18. Reto: genera 100 informes parametrizados.

## 20. Preguntas de reflexión

1. ¿Qué significa parametrizar un informe?
2. ¿Qué parte debe ser constante?
3. ¿Qué parte debe variar?
4. ¿Por qué separar segmentación y generación?
5. ¿Qué aporta un dataset maestro?
6. ¿Qué debe ocurrir si falla un informe?
7. ¿Por qué validar después de guardar?
8. ¿Qué aporta una auditoría?
9. ¿Cómo reutilizarías esto para departamentos?
10. ¿Qué diferencia hay entre este sistema y el generador PDF con LaTeX?

## 21. Cierre

Ahora tenemos dos sistemas de generación masiva:

```text
dataset maestro
   ├── Excel → workbook interactivo parametrizado
   └── LaTeX → PDF final reproducible
```

El siguiente bloque entra en automatización conectada:

```text
HTTP
 ↓
requests
 ↓
APIs REST
 ↓
JSON
 ↓
servicios externos
```